# Vilier API Label Review

Notebook này dùng để nhìn cùng lúc input audio, ASR transcript và output label từ API Qwen/DashScope.

Chạy pipeline trước để có `outputs/<audio_id>/transcript.json`, rồi chạy notebook này.

In [ ]:
from pathlib import Path
import json
import os
import sys

from IPython.display import Audio, display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.labeling import QwenLabelingRunner, DryRunLabelingRunner, label_transcripts

try:
    import pandas as pd
except ImportError:
    pd = None

config = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
labeling_config = config.get("state_labeling", {})
api_key_env = labeling_config.get("api_key_env", "DASHSCOPE_API_KEY")

print("project:", PROJECT_ROOT.name)
print("api_key_env:", api_key_env)
print("api_key_present:", bool(os.environ.get(api_key_env, "")))
print("model:", labeling_config.get("model"))
print("labels:", labeling_config.get("labels"))

## 1. Chọn Output Cần Review

In [ ]:
def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

available = sorted(
    p.parent.name
    for p in (PROJECT_ROOT / "outputs").glob("*/transcript.json")
)
print("available audio_id with transcript.json:")
for item in available:
    print("-", item)

# Đổi AUDIO_ID nếu muốn xem output khác.
AUDIO_ID = available[0] if available else "vi_conv"
LIMIT = 20
print("selected AUDIO_ID:", AUDIO_ID)

## 2. Load Manifest, Input Audio, Transcript

In [ ]:
output_dir = PROJECT_ROOT / "outputs" / AUDIO_ID
manifest_path = output_dir / "manifest.timeline.json"
transcript_path = output_dir / "transcript.json"

if not transcript_path.exists():
    raise FileNotFoundError(f"Missing {rel(transcript_path)}. Run pipeline first, then rerun this notebook.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
transcripts = json.loads(transcript_path.read_text(encoding="utf-8"))

source_audio = PROJECT_ROOT / manifest.get("source_audio", "") if manifest.get("source_audio") else None
standardized_audio = output_dir / manifest.get("standardized_audio", "audio.standardized.wav")

print("output_dir:", rel(output_dir))
print("manifest:", rel(manifest_path), manifest_path.exists())
print("transcript:", rel(transcript_path), len(transcripts))
print("source_audio:", rel(source_audio) if source_audio else "")
print("standardized_audio:", rel(standardized_audio), standardized_audio.exists())

## 3. Nghe Input Audio

In [ ]:
if source_audio and source_audio.exists():
    print("original:", rel(source_audio))
    display(Audio(filename=str(source_audio)))
elif standardized_audio.exists():
    print("standardized:", rel(standardized_audio))
    display(Audio(filename=str(standardized_audio)))
else:
    print("No input audio found for review.")

## 4. Bảng ASR Transcript Hiện Có

In [ ]:
def compact_rows(records):
    rows = []
    for idx, item in enumerate(records):
        rows.append({
            "idx": idx,
            "id": item.get("id", ""),
            "vad_id": item.get("vad_id", ""),
            "speaker": item.get("speaker", ""),
            "start": item.get("start", ""),
            "end": item.get("end", ""),
            "audio": item.get("audio", ""),
            "text": item.get("text", ""),
            "state_label": item.get("state_label", ""),
            "state_confidence": item.get("state_confidence", ""),
            "state_reason": item.get("state_reason", ""),
        })
    return rows

review_records = transcripts[:LIMIT]
rows = compact_rows(review_records)
if pd:
    display(pd.DataFrame(rows))
else:
    display(rows)

## 5. Gọi API Để Tạo Output Label

In [ ]:
# REAL_API=True sẽ gọi Qwen/DashScope thật. REAL_API=False chỉ dùng dry-run để test flow.
REAL_API = True
API_LIMIT = 5

if REAL_API:
    if not os.environ.get(api_key_env, ""):
        raise RuntimeError(f"Missing {api_key_env}. Set it before starting Jupyter.")
    runner = QwenLabelingRunner(labeling_config)
else:
    runner = DryRunLabelingRunner(
        labels=labeling_config.get("labels", ["complete", "incomplete"]),
        model_name=labeling_config.get("model", "qwen-plus"),
    )

api_input_records = transcripts[:API_LIMIT]
labeled_records = label_transcripts(api_input_records, runner)

label_rows = compact_rows(labeled_records)
if pd:
    display(pd.DataFrame(label_rows))
else:
    display(label_rows)

## 6. Nghe Từng Utterance Và Nhìn Transcript + Label

In [ ]:
def show_utterance(index=0, use_api_labels=True):
    records = labeled_records if use_api_labels and "labeled_records" in globals() else transcripts
    item = records[index]
    audio_path = output_dir / item.get("audio", "")
    print("idx:", index)
    print("id:", item.get("id", ""))
    print("time:", item.get("start", ""), "->", item.get("end", ""))
    print("speaker:", item.get("speaker", ""))
    print("audio:", rel(audio_path))
    print("ASR:", item.get("text", ""))
    print("label:", item.get("state_label", ""))
    print("confidence:", item.get("state_confidence", ""))
    print("reason:", item.get("state_reason", ""))
    if audio_path.exists():
        display(Audio(filename=str(audio_path)))
    else:
        print("Missing utterance audio file.")

# Đổi index để nghe/xem utterance khác.
show_utterance(0)

## 7. Lưu Label Review Ra File

In [ ]:
review_dir = PROJECT_ROOT / "logs" / "api_tests" / AUDIO_ID
review_dir.mkdir(parents=True, exist_ok=True)
review_path = review_dir / "label_review.json"

payload = {
    "audio_id": AUDIO_ID,
    "source_audio": rel(source_audio) if source_audio else "",
    "transcript_path": rel(transcript_path),
    "model": labeling_config.get("model"),
    "labels": labeling_config.get("labels"),
    "records": labeled_records if "labeled_records" in globals() else review_records,
}
review_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(review_path))

In [2]:
from diarizen.pipelines.inference import DiariZenPipeline

# load pre-trained model
diar_pipeline = DiariZenPipeline.from_pretrained("BUT-FIT/diarizen-wavlm-large-s80-md")
# apply diarization pipeline
diar_results = diar_pipeline('inputs/real.wav')

# print results
for turn, _, speaker in diar_results.itertracks(yield_label=True):
    print(f"start={turn.start:.1f}s stop={turn.end:.1f}s speaker_{speaker}")

# load pre-trained model and save RTTM result
diar_pipeline = DiariZenPipeline.from_pretrained(
        "BUT-FIT/diarizen-wavlm-large-s80-md",
        rttm_out_dir='.'
)
# apply diarization pipeline
diar_results = diar_pipeline('audio.wav', sess_name='session_name')


ModuleNotFoundError: No module named 'diarizen'

## 8. Test speechbrain/sepformer-wsj02mix

Source separation using SpeechBrain. Clone huggingface repo to get `test_mixture.wav`.

In [1]:
!git lfs install
!git clone https://huggingface.co/speechbrain/sepformer-wsj02mix pretrained_models/sepformer-wsj02mix_repo

Updated Git hooks.
Git LFS initialized.
Cloning into 'pretrained_models/sepformer-wsj02mix_repo'...
remote: Enumerating objects: 135, done.
remote: Total 135 (delta 0), reused 0 (delta 0), pack-reused 135 (from 1)
Receiving objects: 100% (135/135), 59.58 KiB | 628.00 KiB/s, done.
Resolving deltas: 100% (76/76), done.
Filtering content: 100% (4/4), 107.90 MiB | 7.54 MiB/s, done.


In [2]:
from speechbrain.inference.separation import SepformerSeparation as separator
import torchaudio

model = separator.from_hparams(source="speechbrain/sepformer-wsj02mix", savedir='pretrained_models/sepformer-wsj02mix')

# Use test_mixture.wav from the downloaded repo
est_sources = model.separate_file(path='pretrained_models/sepformer-wsj02mix_repo/test_mixture.wav') 

torchaudio.save("source1hat.wav", est_sources[:, :, 0].detach().cpu(), 8000)
torchaudio.save("source2hat.wav", est_sources[:, :, 1].detach().cpu(), 8000)
print("Saved source1hat.wav and source2hat.wav")

from IPython.display import Audio, display
print("Source 1:")
display(Audio(filename="source1hat.wav"))
print("Source 2:")
display(Audio(filename="source2hat.wav"))


/opt/anaconda3/envs/sommelier/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved source1hat.wav and source2hat.wav
Source 1:


Source 2:
